# 01 · Terminology mapping

Map free-text expressions to **standard OMOP concepts**. This is the core of AATM; every
later example reuses it.

A `TerminologyMapper` chains four steps: **translate** (optional) → **retrieve**
candidate concepts from the vector index → **rerank** (optional) → **select** one.

**Before you start:** run `aatm init` (it builds `.aatm/`) and set `OPENAI_API_KEY` in `.env`.
This notebook uses the `text-embedding-3-small` index; use the embedding model you chose in
`aatm init`.

In [1]:
# Run from the folder where you ran `aatm init` (it holds .aatm/ and .env).
import os

if not os.path.isdir(".aatm") and os.path.isdir("../.aatm"):
    os.chdir("..")

## 1. The simplest mapper

Only a retriever is required. The default selector takes the closest candidate.
`save_to_disk=False` keeps results in memory instead of writing a CSV.

Look at the `domain_id` column: the closest match for "heart attack" and "Hb" is a LOINC
*answer* (`Meas Value`), not a condition or a test. Sections 2 and 3 fix this.

In [2]:
import pandas as pd

from aatm.omop import SqliteConceptLookup
from aatm.terminology_mapper import TerminologyMapper

vocabulary = SqliteConceptLookup()  # the local vocabulary database, to show concept names


def results(df):
    # The mapped concepts, with their names.
    ids = pd.to_numeric(df["target_concept_id"]).astype("Int64")
    names = vocabulary.by_id(ids.dropna().tolist())
    df["concept_name"] = [names[i].concept_name if pd.notna(i) else None for i in ids]
    return df[["source_code_description", "target_concept_id", "concept_name", "target_vocabulary_id", "domain_id", "confidence_score"]]


mapper = TerminologyMapper(retriever="text-embedding-3-small")

expressions = ["type 2 diabetes", "heart attack", "Hb", "dor no peito"]
results(mapper.map(expressions, save_to_disk=False))

,source_code_description,target_concept_id,concept_name,target_vocabulary_id,domain_id,confidence_score
0,type 2 diabetes,201826,Type 2 diabetes mellitus,SNOMED,Condition,0.794955
1,heart attack,36210764,Heart attack,LOINC,Meas Value,0.864886
2,Hb,45880609,Hb H,LOINC,Meas Value,0.796920
3,dor no peito,77670,Chest pain,SNOMED,Condition,0.427102


## 2. Let an LLM choose among candidates

An LLM selector reads the top candidates and picks the best one. `top_k` sets how many
it sees. Each text costs one LLM call.

In [3]:
from aatm.selectors import OpenAILLMSelector

llm_mapper = TerminologyMapper(
    retriever="text-embedding-3-small",
    selector=OpenAILLMSelector("gpt-5-mini"),
)
llm_mapper.retriever.top_k = 30

results(llm_mapper.map(expressions, save_to_disk=False))

,source_code_description,target_concept_id,concept_name,target_vocabulary_id,domain_id,confidence_score
0,type 2 diabetes,201826,Type 2 diabetes mellitus,SNOMED,Condition,0.794955
1,heart attack,4329847,Myocardial infarction,SNOMED,Condition,0.864812
2,Hb,4153000,Hemoglobin level estimation,SNOMED,Measurement,0.498251
3,dor no peito,77670,Chest pain,SNOMED,Condition,0.427102


## 3. Restrict the search

"Hb" above became the general SNOMED *Hemoglobin level estimation*. `where` filters the candidates by
metadata (`std_domain_id`, `std_vocabulary_id`), here LOINC lab tests only. The OMOP table
builders in the next notebooks use the same filter to keep each column in its domain.

In [4]:
results(
    llm_mapper.map(
        ["Hb", "serum creatinine"],
        save_to_disk=False,
        where={"$and": [{"std_domain_id": "Measurement"}, {"std_vocabulary_id": "LOINC"}]},
    )
)

,source_code_description,target_concept_id,concept_name,target_vocabulary_id,domain_id,confidence_score
0,Hb,3000963,Hemoglobin [Mass/volume] in Blood,LOINC,Measurement,0.304537
1,serum creatinine,3016723,Creatinine [Mass/volume] in Serum or Plasma,LOINC,Measurement,0.359390


## Next

- Map a whole CSV: `mapper.map(file_path="source_to_concept_map.csv")`, or use the CLI
  (`aatm map`). See the [Python API quick start](https://precisiondata.github.io/aatm/getting-started/quick_start_python_api/).
- Turn clinical notes into OMOP rows: [02_omop_quickstart.ipynb](02_omop_quickstart.ipynb).